# Options-IV Forecasting on Colab T4

Trains two models on `gauss314/options-IV-SP500`:
- **Run A**: per-stock (constituent-level) next-day IV forecasting.
- **Run B**: synthetic equal-weighted S&P 500 index-level next-day IV forecasting.

Uses a single Colab **T4 GPU** (standard CUDA path, no TPU/XLA).

## Setup

In [ ]:
!git clone https://github.com/prathamkul007-max/IV_mech_interp_model.git
%cd IV_mech_interp_model
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## Hugging Face Hub token

Needed only if you want to push trained checkpoints to the Hub (`--push-to-hub` below). The token is entered interactively via `getpass` so it is never written into this notebook file or committed to the repo. Get a token (with write access) from https://huggingface.co/settings/tokens.

In [ ]:
import getpass
import os

os.environ['HF_TOKEN'] = getpass.getpass('Enter your Hugging Face token (leave blank to skip pushing to the Hub): ')

## Run A: constituent-level (per-stock) model

In [ ]:
!python scripts/prepare_options_iv.py \
    --output-dir options_iv_data \
    --seq-length 32 \
    --stride 5

In [ ]:
import json
import numpy as np

train_data = np.load('options_iv_data/train.npz')
print('train X shape:', train_data['X'].shape)
print('train Y shape:', train_data['Y'].shape)
with open('options_iv_data/scaler.json') as f:
    scaler = json.load(f)
print('num features:', len(scaler['mean']))
print('target indices:', scaler['target_indices'])
print('feature names:', scaler['feature_names'])

In [ ]:
!python run_pretrain_iv.py \
    --train-file options_iv_data/train.npz \
    --valid-file options_iv_data/valid.npz \
    --scaler-file options_iv_data/scaler.json \
    --checkpoints-dir checkpoints_iv \
    --d-model 928 \
    --num-layers 12 \
    --num-heads 8 \
    --d-ff 3712 \
    --seq-length 32 \
    --dropout 0.1 \
    --train-batch-size 256 \
    --eval-batch-size 256 \
    --learning-rate 3e-4 \
    --train-steps 5000 \
    --valid-steps 50 \
    --valid-interval 250 \
    --save-interval 250 \
    --saved-checkpoint-limit 3 \
    --mixed-precision fp16 \
    --push-to-hub \
    --hub-repo-id Pratham007xo/iv-forecast-constituent-124m

### Inference/demo: predict tomorrow's IV for a held-out ticker

In [ ]:
import glob
import torch

from gpt2.iv_model import IVModel, IVModelConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

latest_checkpoint = sorted(
    glob.glob('checkpoints_iv/iv_model-*.pt'),
    key=lambda p: int(p.split('-')[-1][:-3]),
)[-1]
print('Loading', latest_checkpoint)
state = torch.load(latest_checkpoint, map_location=device)
model_config = IVModelConfig(**state['config'])
model = IVModel(model_config).to(device)
model.load_state_dict(state['model'])
model.eval()

valid_data = np.load('options_iv_data/valid.npz')
sample_idx = 0
inputs = torch.from_numpy(valid_data['X'][sample_idx:sample_idx + 1]).float().to(device)
targets = valid_data['Y'][sample_idx]

with torch.no_grad():
    predictions = model(inputs).cpu().numpy()[0]

mean = np.asarray(scaler['mean'])[scaler['target_indices']]
std = np.asarray(scaler['std'])[scaler['target_indices']]
predictions_unscaled = predictions * std + mean
targets_unscaled = targets * std + mean

print('Predicted next-day IV buckets (last day of window):', predictions_unscaled[-1])
print('Actual next-day IV buckets (last day of window):   ', targets_unscaled[-1])

### Push constituent-level model to Hugging Face Hub

Training already pushes automatically (`--push-to-hub` in the cell above), but you can also push explicitly/again here — e.g. if you re-ran the training cell without that flag, or just want to confirm the push. Uses the `HF_TOKEN` you entered earlier via `getpass`.

In [ ]:
from gpt2.hub_utils import push_model_to_hub

push_model_to_hub(
    checkpoint_path=latest_checkpoint,
    scaler_file='options_iv_data/scaler.json',
    repo_id='Pratham007xo/iv-forecast-constituent-124m',
)

In [ ]:
import matplotlib.pyplot as plt

atm_idx = scaler['feature_names'][scaler['target_indices'][3]]  # ATM bucket, per detect_target_columns sort order
plt.plot(predictions_unscaled[:, 3], label='predicted ATM IV')
plt.plot(targets_unscaled[:, 3], label='actual ATM IV')
plt.xlabel('day in window')
plt.ylabel('ATM IV')
plt.legend()
plt.title('Constituent-level: predicted vs actual next-day ATM IV')
plt.show()

## Run B: synthetic index-level (equal-weighted S&P 500) model

In [ ]:
!python scripts/prepare_options_iv_index.py \
    --output-dir options_iv_index_data \
    --seq-length 32 \
    --stride 1

In [ ]:
train_data_index = np.load('options_iv_index_data/train.npz')
print('train X shape:', train_data_index['X'].shape)
print('train Y shape:', train_data_index['Y'].shape)

In [ ]:
!python run_pretrain_iv.py \
    --train-file options_iv_index_data/train.npz \
    --valid-file options_iv_index_data/valid.npz \
    --scaler-file options_iv_index_data/scaler.json \
    --checkpoints-dir checkpoints_iv_index \
    --d-model 928 \
    --num-layers 12 \
    --num-heads 8 \
    --d-ff 3712 \
    --seq-length 32 \
    --dropout 0.1 \
    --train-batch-size 64 \
    --eval-batch-size 64 \
    --learning-rate 3e-4 \
    --train-steps 2000 \
    --valid-steps 20 \
    --valid-interval 100 \
    --save-interval 100 \
    --saved-checkpoint-limit 3 \
    --mixed-precision fp16 \
    --push-to-hub \
    --hub-repo-id Pratham007xo/iv-forecast-index-124m

### Inference/demo: predict tomorrow's synthetic index IV

In [ ]:
with open('options_iv_index_data/scaler.json') as f:
    scaler_index = json.load(f)

latest_checkpoint_index = sorted(
    glob.glob('checkpoints_iv_index/iv_model-*.pt'),
    key=lambda p: int(p.split('-')[-1][:-3]),
)[-1]
print('Loading', latest_checkpoint_index)
state_index = torch.load(latest_checkpoint_index, map_location=device)
model_config_index = IVModelConfig(**state_index['config'])
model_index = IVModel(model_config_index).to(device)
model_index.load_state_dict(state_index['model'])
model_index.eval()

valid_data_index = np.load('options_iv_index_data/valid.npz')
inputs_index = torch.from_numpy(valid_data_index['X']).float().to(device)
targets_index = valid_data_index['Y']

with torch.no_grad():
    predictions_index = model_index(inputs_index).cpu().numpy()

mean_index = np.asarray(scaler_index['mean'])[scaler_index['target_indices']]
std_index = np.asarray(scaler_index['std'])[scaler_index['target_indices']]
predictions_index_unscaled = predictions_index[:, -1, :] * std_index + mean_index
targets_index_unscaled = targets_index[:, -1, :] * std_index + mean_index

plt.plot(predictions_index_unscaled[:, 3], label='predicted ATM IV (index)')
plt.plot(targets_index_unscaled[:, 3], label='actual ATM IV (index)')
plt.xlabel('validation window index (time-ordered)')
plt.ylabel('ATM IV')
plt.legend()
plt.title('Synthetic S&P 500 index-level: predicted vs actual next-day ATM IV')
plt.show()

### Push index-level model to Hugging Face Hub

Same as above — training already pushes automatically, this is for pushing explicitly/again.

In [ ]:
from gpt2.hub_utils import push_model_to_hub

push_model_to_hub(
    checkpoint_path=latest_checkpoint_index,
    scaler_file='options_iv_index_data/scaler.json',
    repo_id='Pratham007xo/iv-forecast-index-124m',
)

## Push both models together

Run this once both Run A and Run B have finished training. It looks up each run's latest checkpoint directly from disk (`checkpoints_iv/`, `checkpoints_iv_index/`), so it works standalone even after a runtime restart — you don't need to have run the inference cells above first, just re-enter your `HF_TOKEN` if the runtime restarted.

In [ ]:
import glob
import os

from gpt2.hub_utils import push_model_to_hub

runs = [
    {
        'name': 'constituent-level',
        'checkpoints_dir': 'checkpoints_iv',
        'scaler_file': 'options_iv_data/scaler.json',
        'repo_id': 'Pratham007xo/iv-forecast-constituent-124m',
    },
    {
        'name': 'index-level',
        'checkpoints_dir': 'checkpoints_iv_index',
        'scaler_file': 'options_iv_index_data/scaler.json',
        'repo_id': 'Pratham007xo/iv-forecast-index-124m',
    },
]

for run in runs:
    checkpoints = sorted(
        glob.glob(os.path.join(run['checkpoints_dir'], 'iv_model-*.pt')),
        key=lambda p: int(p.split('-')[-1][:-3]),
    )
    if not checkpoints:
        print(f"Skipping {run['name']}: no checkpoint found in {run['checkpoints_dir']}/ (train it first)")
        continue

    latest = checkpoints[-1]
    print(f"Pushing {run['name']} ({latest}) to {run['repo_id']}...")
    push_model_to_hub(
        checkpoint_path=latest,
        scaler_file=run['scaler_file'],
        repo_id=run['repo_id'],
    )

print('Done.')